# Train the Bloop model on a free Colab T4

Fine-tunes **Qwen/Qwen3.5-0.8B** with a light LoRA on the Bloop dataset, merges it, converts it to GGUF, quantizes to **Q4_K_M** (~500 MB), evaluates it against the base model and (optionally) publishes it to Hugging Face.

Run the cells top to bottom. Expected time on a T4: roughly 15-25 minutes for training plus 10-15 minutes for the llama.cpp build/convert/eval. Colab sessions are ephemeral, so download or publish the result before closing the tab.

> This notebook was written without being run end to end. If a cell fails, the error message and `ml/README.md` (troubleshooting) are the first stops.

## 1. Check the runtime is a GPU

Use **Runtime -> Change runtime type -> T4 GPU**. The T4 has no bfloat16, so the training script uses fp16 autocast on CUDA.

In [ ]:
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    raise SystemExit("No GPU found. Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")
print("GPU:", out.stdout.strip())

## 2. Get the training kit

Clones the Bloop repo (for `ml/train/*`, `ml/MODEL_CARD.md`). Change `REPO_URL` if you use a fork. If you cannot clone, upload the `ml/` folder as a zip instead and unzip it to `/content/StudyPet/ml`.

In [ ]:
REPO_URL = "https://github.com/Bndrwe/StudyPet.git"   # change for a fork
import os
if not os.path.isdir("/content/StudyPet"):
    !git clone --depth 1 {REPO_URL} /content/StudyPet
%cd /content/StudyPet
!ls ml ml/train

## 3. Upload the dataset

Choose `train.jsonl` and `eval.jsonl` (made by the dataset step, one `{"messages": [...]}` object per line). If they already exist in the repo checkout (`ml/data/`), this cell keeps them.

In [ ]:
import os, shutil
os.makedirs("ml/data", exist_ok=True)
if not (os.path.exists("ml/data/train.jsonl") and os.path.exists("ml/data/eval.jsonl")):
    from google.colab import files
    for name, data in files.upload().items():
        shutil.move(name, os.path.join("ml/data", name))
for f in ("train.jsonl", "eval.jsonl"):
    p = os.path.join("ml/data", f)
    assert os.path.exists(p), f"missing {p}: upload train.jsonl and eval.jsonl"
    print(f, sum(1 for _ in open(p, encoding="utf-8")), "rows")

## 4. Install

Colab already ships PyTorch, so we install everything else (`torch` is excluded to keep the GPU build). We also try `flash-linear-attention`: Qwen3.5 mixes Gated DeltaNet (linear attention) layers with normal attention, and this package gives them a fast CUDA path. It is optional; without it training still works with the slower PyTorch fallback.

In [ ]:
!grep -v '^torch' ml/train/requirements.txt > /tmp/req.txt && pip install -q -U -r /tmp/req.txt
!pip install -q flash-linear-attention || echo "flash-linear-attention not installed - slower fallback will be used (fine)"
!apt-get -qq install -y cmake > /dev/null
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))

## 4b. Choose the base model

`BASE` is the checkpoint the LoRA starts from. The default is the plain **Qwen/Qwen3.5-0.8B**. Set it to `MIT_BASE` to train on top of MIT CSAIL's gradient-free weight edit ([blog](https://rikiyat.github.io/blog/weight-space-programming/), [checkpoint](https://huggingface.co/RikiyaT/Qwen3.5-0.8B-weight-space-programming), Apache-2.0). That edit removed a looping direction from two MLPs (layers 15 and 19); the rest is identical to Qwen. With `MIT_BASE`, section 6 also converts MIT's checkpoint so the eval compares base / mit-edit / tuned. See `ml/wsp/README.md` for editing weights yourself.


In [ ]:
QWEN_BASE = "Qwen/Qwen3.5-0.8B"
MIT_BASE = "RikiyaT/Qwen3.5-0.8B-weight-space-programming"
BASE = QWEN_BASE   # or MIT_BASE
print("LoRA base:", BASE)


## 5. Time a few steps, then train and merge

The first command times a few real steps and prints an ETA, then stops. The second runs the full training (2 epochs, LoRA r=16, effective batch 8, lr 2e-4 cosine), logs train/eval loss, saves the adapter to `ml/out/adapter` and the merged model to `ml/out/merged`.

If you run out of GPU memory, add `--max-len 768 --grad-checkpointing`.

In [ ]:
!python ml/train/train_lora.py --base {BASE} --dry-run

In [ ]:
!python ml/train/train_lora.py --base {BASE}

## 6. Convert to GGUF and quantize (Q4_K_M)

Clones and builds llama.cpp into `ml/cache/` (CPU build, a few minutes), converts the merged model to f16 GGUF and quantizes it. The last lines show the file size and SHA-256. The base model is converted too so the evaluation can compare the two.

In [ ]:
!bash ml/train/convert_quantize.sh --src ml/out/merged --name bloop

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3.5-0.8B", local_dir="ml/cache/base", allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja", "*.model"])
!bash ml/train/convert_quantize.sh --src ml/cache/base --name base
if BASE == MIT_BASE:  # also convert MIT's edited checkpoint so the eval compares base / mit-edit / tuned
    snapshot_download(MIT_BASE, local_dir="ml/cache/mit", allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja", "*.model"])
    !bash ml/train/convert_quantize.sh --src ml/cache/mit --name mit


## 7. Quick test with llama-cli

A single prompt to see that the quantized model answers sensibly. (If your llama.cpp build uses different flags, check `llama-cli -h`.)

In [ ]:
!ml/cache/llama.cpp/build/bin/llama-cli -m ml/out/bloop-qwen3.5-0.8b-Q4_K_M.gguf -st -n 60 --temp 0.4 -p "I have been on YouTube for 4 minutes instead of studying. Write one short, kind nudge (max 60 characters)." < /dev/null 2>/dev/null | tail -n 5

## 8. Evaluate: base vs tuned

Runs the eval set through `llama-server`, with and without a JSON-schema grammar, and writes `ml/eval/REPORT.md` (plan JSON parse / schema-valid rate, rule checks, nudge length, latency). The Colab build is CPU-only, so `--limit 25` keeps it to a few minutes. Remove it to use the whole eval set.

If `ml/out/mit-qwen3.5-0.8b-Q4_K_M.gguf` (MIT's edit) or `ml/out/bloop-edit-qwen3.5-0.8b-Q4_K_M.gguf` (tuned + a weight-space edit from `ml/wsp/session.ipynb`) exist, they are added to the comparison automatically.


In [ ]:
!python ml/train/eval.py --limit 25
print(open("ml/eval/REPORT.md").read()[:6000])

## 9. Publish to Hugging Face (optional)

`login()` asks for a **write** token (https://huggingface.co/settings/tokens). The cell creates `<your-user>/bloop-qwen3.5-0.8b-GGUF`, uploads the GGUF and the model card as `README.md`, then prints the three values the Bloop app needs: `BLOOP_MODEL_URL`, `BLOOP_MODEL_SHA256` and `BLOOP_MODEL_BYTES` (constants in `src/main/ai/local/bloop.ts`).

In [ ]:
import hashlib, os
from huggingface_hub import HfApi, create_repo, login, upload_file

GGUF = "ml/out/bloop-qwen3.5-0.8b-Q4_K_M.gguf"
login()
api = HfApi()
user = api.whoami()["name"]
repo_id = f"{user}/bloop-qwen3.5-0.8b-GGUF"
create_repo(repo_id, repo_type="model", exist_ok=True)
fname = os.path.basename(GGUF)
upload_file(path_or_fileobj=GGUF, path_in_repo=fname, repo_id=repo_id, commit_message=f"Add {fname}")
upload_file(path_or_fileobj="ml/MODEL_CARD.md", path_in_repo="README.md", repo_id=repo_id, commit_message="Model card")

h = hashlib.sha256()
with open(GGUF, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print()
print("Put these in src/main/ai/local/bloop.ts:")
print(f"  BLOOP_MODEL_URL    = 'https://huggingface.co/{repo_id}/resolve/main/{fname}'")
print(f"  BLOOP_MODEL_SHA256 = '{h.hexdigest()}'")
print(f"  BLOOP_MODEL_BYTES  = {os.path.getsize(GGUF)}")

## Alternative: download the .gguf directly

Skip publishing and just save the file to your computer, then import it in Bloop via **Settings -> AI -> Bloop model -> "Load from file"**. The cell also prints the SHA-256 and size.

In [ ]:
import hashlib, os
from google.colab import files
GGUF = "ml/out/bloop-qwen3.5-0.8b-Q4_K_M.gguf"
h = hashlib.sha256(open(GGUF, "rb").read()).hexdigest()
print("sha256:", h, "| bytes:", os.path.getsize(GGUF))
files.download(GGUF)